# Task 1 ensemble

Every sub-model cell trains one model and stores it in `trained_models/task1/<model>.pt`.
The blend cells only need the stored files, so after a kernel restart nothing has to be retrained.
All hyperparameters: `submission/lib/config.py`, Task 1 section.

In [1]:
%load_ext autoreload
%autoreload 2

# assignment1.py and lib/ live in ./submission (this notebook lives in assignment1_2026)
import sys
from pathlib import Path
sys.path.append(str(Path.cwd() / "submission"))

import torch

import assignment1
from lib.models.bias_model import load_bias_model, store_bias_model
from lib.models.blender import load_blender, store_blender
from lib.config import TASK1_LIGHTGBM_BLENDER, TASK1_RIDGE_BLENDER, TASK1_SUBMITTED_BLENDER
from lib.models.factorization_machine import load_factorization_machine, store_factorization_machine
from lib.models.item_knn import load_item_knn, store_item_knn
from lib.models.SVDpp import load_svd, load_svdpp, store_svd, store_svdpp
from lib.models.zero_model import load_zero_model, store_zero_model

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

2.14.1+cu130 | CUDA available: True | NVIDIA GeForce RTX 5080


### Data and validation
Ratings $\mathcal R = \{(u, i, r_{ui})\}$, $r_{ui} \in [0, 5]$, split at random into 90 % training $\mathcal T$ and 10 % validation $\mathcal V$ (`VALIDATION_FRACTION`, `SPLIT_SEED`).
Every sub-model: fit on $\mathcal T$ $\rightarrow$ predictions $\hat r^{(m)}_{ui}$ for $\mathcal V$ (input of the blender) $\rightarrow$ refit on $\mathcal T \cup \mathcal V$ $\rightarrow$ test predictions.

$$\mathrm{MSE} = \frac{1}{|\mathcal V|}\sum_{(u,i)\in\mathcal V}\big(r_{ui}-\hat r_{ui}\big)^2$$

In [2]:
task1 = assignment1.load_task1()

## Sub-models

### Bias model (PML Eq. 5.10 without factors)
$$\hat r_{ui} = \mu + b_u + b_i$$
Closed form, alternating for 10 rounds ($U_i$ = users who rated $i$, $I_u$ = items rated by $u$, $\lambda$ = `regularization`):
$$b_i \leftarrow \frac{\sum_{u\in U_i}\left(r_{ui}-\mu-b_u\right)}{|U_i|+\lambda}, \qquad b_u \leftarrow \frac{\sum_{i\in I_u}\left(r_{ui}-\mu-b_i\right)}{|I_u|+\lambda}$$

In [3]:
# bias model: mu + b_u + b_i in closed form (seconds)
bias_model_result = assignment1.train_task1_bias_model(task1)
store_bias_model(bias_model_result, "task1")

[22:02:13] bias model | valid MSE 0.7647
[22:02:13] stored bias_model -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task1\bias_model.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/bias_model.pt')

### SVD / latent factor model (PML Eq. 5.10)
$$\hat r_{ui} = \mu + b_u + b_i + \mathbf q_i^\top \mathbf p_u$$
$$\min_{b,\mathbf p,\mathbf q}\ \sum_{(u,i)\in\mathcal T}\big(r_{ui}-\hat r_{ui}\big)^2 + \lambda_b\Big(\sum_u b_u^2+\sum_i b_i^2\Big) + \lambda_f\Big(\sum_u\lVert\mathbf p_u\rVert^2+\sum_i\lVert\mathbf q_i\rVert^2\Big)$$
Adam on mini-batches (each parameter's penalty is spread over the ratings that use it), biases start at the closed-form bias model,
early stopping on the validation MSE, then a refit on all ratings for the best number of epochs.

In [4]:
# SVD (a few minutes on the RTX 5080)
svd_result = assignment1.train_task1_svd(task1)
store_svd(svd_result, "task1")

[22:02:15] epoch   0 | train MSE 0.6996 | valid MSE 0.7647  (starting point)
[22:02:21] epoch   1 | train MSE 0.6983 | valid MSE 0.7644
[22:02:27] epoch   2 | train MSE 0.6913 | valid MSE 0.7628
[22:02:34] epoch   3 | train MSE 0.6732 | valid MSE 0.7589
[22:02:40] epoch   4 | train MSE 0.6480 | valid MSE 0.7553
[22:02:46] epoch   5 | train MSE 0.6192 | valid MSE 0.7525
[22:02:53] epoch   6 | train MSE 0.5890 | valid MSE 0.7509
[22:02:59] epoch   7 | train MSE 0.5590 | valid MSE 0.7502
[22:03:06] epoch   8 | train MSE 0.5304 | valid MSE 0.7503
[22:03:12] epoch   9 | train MSE 0.5038 | valid MSE 0.7510
[22:03:19] epoch  10 | train MSE 0.4795 | valid MSE 0.7521
[22:03:25] epoch  11 | train MSE 0.4575 | valid MSE 0.7534
[22:03:31] epoch  12 | train MSE 0.4378 | valid MSE 0.7549
[22:03:38] epoch  13 | train MSE 0.4202 | valid MSE 0.7565
[22:03:44] epoch  14 | train MSE 0.4047 | valid MSE 0.7581
[22:03:51] epoch  15 | train MSE 0.3909 | valid MSE 0.7597
[22:03:57] epoch  16 | train MSE 0.378

WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/svd.pt')

### SVD++ (Koren 2008; implicit feedback as in FISM, PML Sec. 5.3.2)
$$\hat r_{ui} = \mu + b_u + b_i + \mathbf q_i^\top\Big(\mathbf p_u + |N(u)|^{-1/2}\sum_{j\in N(u)}\mathbf y_j\Big)$$
$N(u)$ = the items $u$ rated in the data the model trains on; $\mathbf y_j$ is regularised with $\lambda_f$ like $\mathbf q_j$. Training as for SVD.

In [5]:
# SVD++ (the slowest sub-model, ~7 min on the RTX 5080)
svdpp_result = assignment1.train_task1_svdpp(task1)
store_svdpp(svdpp_result, "task1")

[22:05:59] epoch   0 | train MSE 0.6996 | valid MSE 0.7647  (starting point)
[22:06:39] epoch   1 | train MSE 0.6639 | valid MSE 0.7567
[22:07:18] epoch   2 | train MSE 0.5687 | valid MSE 0.7467
[22:07:57] epoch   3 | train MSE 0.4498 | valid MSE 0.7445
[22:08:35] epoch   4 | train MSE 0.3510 | valid MSE 0.7482
[22:09:15] epoch   5 | train MSE 0.2784 | valid MSE 0.7543
[22:09:55] epoch   6 | train MSE 0.2280 | valid MSE 0.7603
[22:10:34] epoch   7 | train MSE 0.1933 | valid MSE 0.7660
[22:11:14] epoch   8 | train MSE 0.1694 | valid MSE 0.7712
[22:11:53] epoch   9 | train MSE 0.1532 | valid MSE 0.7755
[22:12:33] epoch  10 | train MSE 0.1416 | valid MSE 0.7787
[22:13:12] epoch  11 | train MSE 0.1333 | valid MSE 0.7825
[22:13:52] epoch  12 | train MSE 0.1270 | valid MSE 0.7850
[22:14:32] epoch  13 | train MSE 0.1221 | valid MSE 0.7872
[22:15:12] epoch  14 | train MSE 0.1186 | valid MSE 0.7895
[22:15:51] epoch  15 | train MSE 0.1156 | valid MSE 0.7909
[22:16:31] epoch  16 | train MSE 0.113

WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/svdpp.pt')

### Item-item kNN on the bias model's residuals (PML Ch. 4; Koren 2010)
Baseline $b_{ui} = \mu + b_u + b_i$ (closed form), residuals $e_{uj} = r_{uj} - b_{uj}$.

Shrunk cosine similarity ($n_{ij}$ = users who rated both, $\lambda_s$ = `similarity_shrinkage`), only the $k$ most similar items $N_k(i)$ are kept:
$$s_{ij} = \frac{\sum_u e_{ui}\,e_{uj}}{\lVert\mathbf e_{\cdot i}\rVert\,\lVert\mathbf e_{\cdot j}\rVert}\cdot\frac{n_{ij}}{n_{ij}+\lambda_s}$$
$$\hat r_{ui} = b_{ui} + \frac{\sum_{j\in N_k(i)\cap I_u} s_{ij}\,e_{uj}}{\sum_{j\in N_k(i)\cap I_u} s_{ij} + \lambda_p}$$

In [6]:
# item-item kNN on the bias model's residuals (CPU, a few minutes)
item_knn_result = assignment1.train_task1_item_knn(task1)
store_item_knn(item_knn_result, "task1")

[22:23:40] item kNN | 2,549,614 neighbour links | valid MSE 0.7564
[22:24:02] stored item_knn -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task1\item_knn.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/item_knn.pt')

### Zero model: $P(r_{ui} = 0)$ (logistic regression, PML Sec. 3.3)
$$P(r_{ui}=0) = \sigma\big(w_0 + \mathbf w^\top \mathbf x_{ui}\big), \qquad \mathbf x_{ui} = \Big[\operatorname{logit}\tilde z_u,\ \operatorname{logit}\tilde z_i,\ \log(1+n_u),\ \log(1+n_i)\Big]$$
$$\tilde z_u = \frac{z_u + s_u\,\pi}{n_u + s_u}$$
$z_u$ = zeros of user $u$, $n_u$ = ratings of $u$ (likewise for item $i$ with $s_i$), $\pi$ = global share of zeros, $s$ = prior strength.
Training rows leave their own rating out; features standardised. The output is a probability, an input of the blender (not a rating).

In [1]:
# zero model: P(rating = 0) (CPU, under a minute)
zero_model_result = assignment1.train_task1_zero_model(task1)
store_zero_model(zero_model_result, "task1")

NameError: name 'assignment1' is not defined

### Factorization machine (Rendle 2010; PML Eq. 6.3)
$$\hat r(\mathbf x) = \mu + \sum_f w_f x_f + \sum_{f<g}\langle\mathbf v_f,\mathbf v_g\rangle\,x_f x_g$$
One active feature per field: user $u$, item $i$, the user's activity bucket $a_u$ and the item's popularity bucket $a_i$ (deciles of #ratings):
$$\hat r_{ui} = \mu + w_u + w_i + w_{a_u} + w_{a_i} + \sum_{f<g\ \in\ \{u,\,i,\,a_u,\,a_i\}}\langle\mathbf v_f,\mathbf v_g\rangle$$
Trained like SVD (Adam, L2 penalty, early stopping, refit).

In [8]:
# factorization machine (a few minutes on the RTX 5080)
# not in the blend yet -> add "factorization_machine" to TASK1_BLENDED_MODELS in lib/config.py to use it
factorization_machine_result = assignment1.train_task1_factorization_machine(task1)
store_factorization_machine(factorization_machine_result, "task1")

[22:24:03] epoch   0 | train MSE 0.6996 | valid MSE 0.7647  (starting point)
[22:24:11] epoch   1 | train MSE 0.6959 | valid MSE 0.7632
[22:24:18] epoch   2 | train MSE 0.6886 | valid MSE 0.7610
[22:24:25] epoch   3 | train MSE 0.6770 | valid MSE 0.7588
[22:24:32] epoch   4 | train MSE 0.6577 | valid MSE 0.7566
[22:24:40] epoch   5 | train MSE 0.6312 | valid MSE 0.7553
[22:24:47] epoch   6 | train MSE 0.6005 | valid MSE 0.7564
[22:24:54] epoch   7 | train MSE 0.5692 | valid MSE 0.7594
[22:25:01] epoch   8 | train MSE 0.5389 | valid MSE 0.7640
[22:25:08] epoch   9 | train MSE 0.5111 | valid MSE 0.7696
[22:25:16] epoch  10 | train MSE 0.4857 | valid MSE 0.7754
[22:25:23] epoch  11 | train MSE 0.4626 | valid MSE 0.7811
[22:25:30] epoch  12 | train MSE 0.4416 | valid MSE 0.7864
[22:25:37] epoch  13 | train MSE 0.4226 | valid MSE 0.7921
[22:25:45] epoch  14 | train MSE 0.4054 | valid MSE 0.7969
[22:25:52] epoch  15 | train MSE 0.3896 | valid MSE 0.8011
[22:25:59] epoch  16 | train MSE 0.375

WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/factorization_machine.pt')

## Blend

### Blender input (stacking)
For every validation rating $(u,i)\in\mathcal V$: the predictions $\hat r^{(1)}_{ui},\dots,\hat r^{(M)}_{ui}$ of the sub-models in `blended_models`,
made by models that never saw $\mathcal V$. For the test rows: the predictions of the refitted sub-models.

In [9]:
# stored sub-models of the blend (also works right after a kernel restart)
sub_model_loaders = {
    "bias_model": load_bias_model,
    "svd": load_svd,
    "svdpp": load_svdpp,
    "item_knn": load_item_knn,
    "zero_model": load_zero_model,
    "factorization_machine": load_factorization_machine,
}
task1_model_results = {model_name: sub_model_loaders[model_name]("task1") for model_name in TASK1_SUBMITTED_BLENDER.blended_models}

### Ridge blend (linear stacking)
$$\hat r_{ui} = \operatorname{clip}_{[0,5]}\Big(\theta_0 + \sum_{m=1}^{M}\theta_m\,\hat r^{(m)}_{ui}\Big), \qquad \min_{\boldsymbol\theta}\ \sum_{(u,i)\in\mathcal V}\big(r_{ui} - \theta_0 - \boldsymbol\theta^\top\hat{\mathbf r}_{ui}\big)^2 + \alpha\lVert\boldsymbol\theta\rVert^2$$
Score: 5-fold out-of-fold MSE on $\mathcal V$; the final blender is fitted on all of $\mathcal V$.

In [10]:
# ridge regression: learned weighted average of the sub-model predictions
ridge_blend_result = assignment1.train_task1_blender(task1, task1_model_results, TASK1_RIDGE_BLENDER)
store_blender(ridge_blend_result, "task1")

[22:27:45] bias_model             valid MSE 0.7647
[22:27:45] svd                    valid MSE 0.7502
[22:27:45] svdpp                  valid MSE 0.7445
[22:27:45] item_knn               valid MSE 0.7564
[22:27:45] ridge blend                valid MSE 0.7321  (out-of-fold, 5 folds)
[22:27:45] ridge blend inputs: bias_model -3.05, svd 0.0734, svdpp 0.644, item_knn 3.34, zero_model 0.0249
[22:27:45] stored ridge_blender -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task1\ridge_blender.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/ridge_blender.pt')

### LightGBM blend (gradient-boosted regression trees)
$$\hat r_{ui} = \operatorname{clip}_{[0,5]}\Big(\sum_{t=1}^{T}\eta\,f_t(\mathbf x_{ui})\Big), \qquad \mathbf x_{ui} = \big[\hat r^{(1)}_{ui},\dots,\hat r^{(M)}_{ui},\ \log(1+n_u),\ \log(1+n_i)\big]$$
Each tree $f_t$ is fitted to the negative gradient of the squared error of the trees before it ($T$ = `lightgbm_n_estimators`, $\eta$ = learning rate).
Score: 5-fold out-of-fold MSE on $\mathcal V$.

In [11]:
# gradient-boosted trees (LightGBM) on the sub-model predictions + #ratings of user and item
lightgbm_blend_result = assignment1.train_task1_blender(task1, task1_model_results, TASK1_LIGHTGBM_BLENDER)
store_blender(lightgbm_blend_result, "task1")

[22:27:48] bias_model             valid MSE 0.7647
[22:27:48] svd                    valid MSE 0.7502
[22:27:48] svdpp                  valid MSE 0.7445
[22:27:48] item_knn               valid MSE 0.7564
[22:27:48] lightgbm blend                valid MSE 0.7342  (out-of-fold, 5 folds)
[22:27:49] lightgbm blend inputs: bias_model 1.83e+03, svd 1.44e+03, svdpp 1.7e+03, item_knn 1.5e+03, zero_model 1.76e+03, log_user_ratings 1.94e+03, log_item_ratings 1.83e+03
[22:27:49] stored lightgbm_blender -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task1\lightgbm_blender.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/lightgbm_blender.pt')

### Submission
The test predictions $\hat r_{ui}$ of the blender chosen in `TASK<n>_SUBMITTED_BLENDER` go into the `rating` column.

In [12]:
# the submitted blend (TASK1_SUBMITTED_BLENDER in lib/config.py) writes submission/output/task_1_predictions.csv
submitted_blend_result = load_blender("task1", TASK1_SUBMITTED_BLENDER.method)
assignment1.write_task1_predictions(submitted_blend_result.test_predictions)
print("wrote", assignment1.TASK1_PREDICTION_FILE)

wrote C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\submission\output\task_1_predictions.csv
